In [1]:
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from collections import defaultdict
import random
from typing import List, Dict, Any, Tuple
import ast
import re

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
client = OpenAI(api_key=OPENAI_API_KEY)

In [2]:
# Load the original persona JSONL data
with open("data/페르소나 가중치 변환.jsonl", "r", encoding="utf-8") as f:
    personas = [json.loads(line) for line in f]
    
with open("data/product_info.json", "r", encoding="utf-8") as f:
    product_info_list = json.load(f)

In [3]:
CLUSTER_CATALOG = {
    0: {"label": "실속형 미식가", "description": "편리성을 중시하면서도 새로운 맛과 제품을 시도하는 데 적극적인 소비자 그룹입니다. 가격에 민감하기보다는 효율성과 맛을 동시에 추구합니다."},
    1: {"label": "건강 추구형 소비자", "description": "가격이나 브랜드에 크게 구애받지 않고 건강을 최우선으로 고려하는 프리미엄 소비자 그룹입니다. 건강과 편리성을 모두 만족시키는 제품에 기꺼이 지갑을 엽니다."},
    3: {"label": "트렌드 주도형 소비자", "description": "자신의 취향과 경험을 중시하는 소비자 그룹입니다. 단순히 배를 채우는 것 이상의 가치를 추구하며, 새로운 제품을 가장 먼저 경험하고 공유하려는 경향이 강합니다."},
}

def enforce_cluster_meta(persona: Dict[str, Any]) -> Dict[str, Any]:
    """
    LLM이 생성한 persona에서 meta.cluster에 맞춰
    meta.label과 meta.description을 카탈로그 값으로 강제 세팅.
    """
    meta = persona.get("meta", {})
    cluster = meta.get("cluster")

    # 방어적 캐스팅
    if isinstance(cluster, bool):
        cluster = int(cluster)
    if isinstance(cluster, (int, float)):
        cluster = int(cluster)
    else:
        raise ValueError("meta.cluster must be an integer.")

    if cluster not in CLUSTER_CATALOG:
        raise ValueError(f"Unknown cluster: {cluster}")

    meta["label"] = CLUSTER_CATALOG[cluster]["label"]
    meta["description"] = CLUSTER_CATALOG[cluster]["description"]
    return persona

def cluster_catalog_block() -> str:
    lines = ["[Cluster Catalog: DO NOT DEVIATE]"]
    for k, v in CLUSTER_CATALOG.items():
        lines.append(f"{k}:")
        lines.append(f"  label: \"{v['label']}\"")
        lines.append(f"  description: \"{v['description']}\"")
    return "\n".join(lines)

def meta_constraints_block() -> str:
    return (
        "[Meta Constraints]\n"
        "- \"meta.cluster\"가 정해지면, \"meta.label\"과 \"meta.description\"은 반드시 위 Catalog의 동일한 값을 그대로 복사한다.\n"
        "- 재해석·변형·요약 금지. 오타 금지."
    )

def operation_tips_block() -> str:
    return (
        "[Operation Tips]\n"
        "1) few-shot 안에도 3개의 서로 다른 cluster를 포함해, cluster가 바뀌면 label/description도 달라짐을 암시적으로 학습시킨다.\n"
        "2) 서버 보정(enforce_cluster_meta)을 항상 거친다. 프롬프트만 믿지 않는다.\n"
        "3) 클러스터 설명을 업데이트할 때는 Catalog 한 곳만 수정해 전체 파이프라인의 일관성을 유지한다.\n"
        "4) 예시 및 생성 결과에는 연령대(20대, 30대, 40대, 50대, 60대 이상), 성별, 직업의 다양성이 반영되도록 주의한다."
    )

def sample_diverse_examples(personas: List[Dict[str, Any]], k: int = 10) -> List[Dict[str, Any]]:
    selected, seen_keys = [], set()
    clusters = [0, 1, 3]
    for c in clusters:
        cands = [p for p in personas if p["meta"]["cluster"] == c]
        random.shuffle(cands)
        for p in cands:
            key = (c, p["attributes"]["age"]["value"], p["attributes"]["gender"]["value"], p["attributes"]["job"]["value"])
            if key not in seen_keys:
                selected.append(p)
                seen_keys.add(key)
                break
    remainder = [p for p in personas if p not in selected]
    random.shuffle(remainder)
    for p in remainder:
        key = (p["meta"]["cluster"], p["attributes"]["age"]["value"], p["attributes"]["gender"]["value"], p["attributes"]["job"]["value"])
        if key not in seen_keys:
            selected.append(p)
            seen_keys.add(key)
        if len(selected) >= k:
            break
    return selected

In [4]:
def flatten_persona_for_fewshot(p: dict) -> dict:
    return {
        "persona_key": p["persona_key"],
        "attributes": {k: v["value"] for k, v in p["attributes"].items()},
        "meta": p["meta"]
    }

def build_fewshot_block(personas: List[Dict[str, Any]], k=10) -> str:
    examples = sample_diverse_examples(personas, k)
    block = "다음은 소비자 페르소나 예시입니다:\n\n"
    for p in examples:
        p = enforce_cluster_meta(p)
        flat = {
            "persona_key": p["persona_key"],
            "attributes": {k: v["value"] for k, v in p["attributes"].items()},
            "meta": p["meta"]
        }
        block += "```json\n" + json.dumps(flat, ensure_ascii=False, indent=2) + "\n```\n\n"
    return block

In [5]:
def build_conditioned_prompt(product_name: str, few_shot_block: str, start_index: int, batch_size: int = 5) -> str:    
    return f"""
[Cluster Catalog: DO NOT DEVIATE]
0: {CLUSTER_CATALOG[0]}
1: {CLUSTER_CATALOG[1]}
3: {CLUSTER_CATALOG[3]}

[Constraints]
- 생성되는 {batch_size}명의 페르소나는 cluster 0, 1, 3을 골고루 포함하도록 할 것
- 성별, 연령대, 직업군의 다양성이 유지되어야 하며, 특정 집단으로 편중되지 않도록 유의할 것
- meta.label / meta.description은 반드시 위 Catalog에서 복사할 것 (변형 금지)

{few_shot_block}
이제 위 조건을 만족하는 새로운 페르소나 {batch_size}개를 JSON 배열 형식으로 생성해주세요.

제품 ID: {product_name}
페르소나 번호: {product_name}_{start_index}

[출력 형식 및 속성 예시값] (JSON 배열 형식! 단일 객체 아님)

아래는 페르소나를 생성할 때 따르는 속성 목록과 형식 예시입니다. 단, 속성 값은 반드시 아래 예시값 중에서 다양하게 조합하여 사용하세요.

```json
{{
  "persona_key": "{product_name}_{start_index}",
  "attributes": {{
    "gender": "남자 또는 여자 중 선택",
    "age": "20대 / 30대 / 40대 / 50대 / 60대 이상 중 선택",
    "job": "관리자 / 군인 / 기능원 및 관련 기능 종사자 / 농림어업 숙련 종사자 / 단순노무 종사자 / 사무 종사자 / 서비스 종사자 / 전문가 및 관련 종사자 / 판매 종사자 / 장치·기계 조작 및 조립 종사자 / 주부 / 취업 준비 중 / 학생 중 선택",
    "education": "고졸(대학 재학 포함) / 대학교 졸업(전문대졸/대학원생 포함) / 대학원 졸업 이상 / 중졸 이하 중 선택",
    "region": "강원도 / 경기도 / 경상남도 / 경상북도 / 광주광역시 / 대구광역시 / 대전광역시 / 서울특별시 / 세종특별자치시 / 울산광역시 / 인천광역시 / 전라남도 / 전라북도 / 제주특별자치도 / 충청남도 / 충청북도 중 선택",
    "household": "1인 가구 / 1세대가족 / 2세대가족 중 선택",
    "marriage": "기혼 / 미혼(사별, 이혼 포함) 중 선택",
    "income_status": "맞벌이 함 / 맞벌이 하지 않음 중 선택",
    "income_month": "100만원 미만 ~ 1,000만원 이상 중 선택",

    "brand_loyalty_scaled": "0.0 ~ 1.0 사이의 실수값",
    "cooking_convenience_scaled": "0.0 ~ 1.0 사이의 실수값",
    "health_orientation_scaled": "0.0 ~ 1.0 사이의 실수값",
    "hmr_preference_scaled": "0.0 ~ 1.0 사이의 실수값",
    "premium_orientation_scaled": "0.0 ~ 1.0 사이의 실수값",
    "price_sensitivity_scaled": "0.0 ~ 1.0 사이의 실수값",
    "variety_seeking_scaled": "0.0 ~ 1.0 사이의 실수값"
  }},
  "meta": {{
    "cluster": "0, 1, 3 중 택일",
    "label": "해당 cluster에 맞는 label을 Catalog에서 복사",
    "description": "해당 cluster에 맞는 description을 Catalog에서 복사"
  }},
}}
```"""

In [6]:
random.seed(42)
fewshot_examples = sample_diverse_examples(personas, k=10)
summary = []
for p in fewshot_examples:
    summary.append({
        "persona_key": p.get("persona_key", "-"),
        "cluster": p["meta"]["cluster"],
        "gender": p["attributes"]["gender"]["value"],
        "age": p["attributes"]["age"]["value"],
        "job": p["attributes"]["job"]["value"],
        "region": p["attributes"].get("region", {}).get("value", "-"),
        "income_month": p["attributes"].get("income_month", {}).get("value", "-"),
    })


print(pd.DataFrame(summary).to_string(index=False))

 persona_key  cluster gender    age             job  region income_month
          74        0     여자    40대         취업 준비 중   대구광역시            -
         277        1     여자 60대 이상              주부     강원도            -
         304        3     여자 60대 이상 기능원 및 관련 기능 종사자   인천광역시            -
         240        3     여자    50대              주부   서울특별시            -
          64        3     남자    40대         서비스 종사자   울산광역시            -
          42        3     여자    30대         서비스 종사자   대전광역시            -
         324        1     여자 60대 이상          판매 종사자   부산광역시            -
         125        3     여자    40대          사무 종사자     경기도            -
         178        0     여자    50대          사무 종사자 제주특별자치도            -
         265        0     여자 60대 이상              주부     경기도            -


In [7]:
def generate_monthly_prediction(product_name: str, launch_date: str, base_value: float = 3.0) -> Dict[str, Any]:
    launch = datetime.strptime(launch_date, "%Y.%m.%d")
    start = datetime(2024, 7, 1)
    row = {"product_name": product_name}
    for i in range(12):
        target = datetime(start.year + (start.month + i - 1) // 12, (start.month + i - 1) % 12 + 1, 1)
        col = f"months_since_launch_{i+1}"
        row[col] = 0 if target < launch else round(base_value + i * 0.1, 2)
    return row

In [10]:
# 디렉토리 생성
os.makedirs("data/outputs/llm_outputs", exist_ok=True)
os.makedirs("data/outputs", exist_ok=True)

monthly_rows = []

for idx, product in enumerate(product_info_list):
    product_name = product["product_name"]
    release_date = product["release_date"]
    
    random.seed(42)
    few_shot_block = build_fewshot_block(personas, k=10)
    personas_output = []

    for i in range(0, 20, 5): # 5명씩 batch 생성
        prompt = build_conditioned_prompt(product_name, few_shot_block, start_index=i, batch_size=5)

        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": "다음 조건을 만족하는 페르소나 정보를 JSON 배열 형식(UTF-8, 인코딩된 따옴표 없음)으로 정확하게 생성하세요. 반드시 JSON 배열만 출력하세요."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0
            )

            content = response.choices[0].message.content.strip()
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0].strip()
            elif "```" in content:
                content = content.split("```")[1].strip()

            parsed = json.loads(content)
            for j, persona in enumerate(parsed):
                persona["persona_key"] = f"{product_name}_{i + j}"
                persona = enforce_cluster_meta(persona)
                personas_output.append(persona)

        except Exception as e:
            print(f"[{product_name}] 오류 발생 (batch {i}~{i+4}): {e}")
            continue

    with open(f"data/outputs/llm_outputs/{product_name}_20_personas.jsonl", "w", encoding="utf-8") as f:
        for p in personas_output:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    monthly_rows.append(generate_monthly_prediction(product_name, release_date))

In [11]:
monthly_df = pd.DataFrame(monthly_rows)
monthly_df.to_csv("data/outputs/final_monthly_prediction.csv", index=False, encoding="utf-8-sig")